In [1]:
!git clone https://github.com/pavanndasarii/ai-ml-final-project.git

Cloning into 'ai-ml-final-project'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 42 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 252.89 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [2]:
import os

os.listdir("ai-ml-final-project/data_pipeline")

['README.md', '01_data_pipeline.ipynb']

In [3]:
with open("ai-ml-final-project/data_pipeline/README.md", "r") as file:
    print(file.read())

# Data Pipeline

Module documentation will be added here.



In [4]:
import json

notebook_path = "ai-ml-final-project/data_pipeline/01_data_pipeline.ipynb"

with open(notebook_path, "r", encoding="utf-8") as file:
    notebook = json.load(file)

for i, cell in enumerate(notebook["cells"]):
    print(f"\n--- Cell {i+1} ({cell['cell_type']}) ---")
    print("".join(cell["source"]))


--- Cell 1 (markdown) ---
# Book Scraping Project

This project scrapes books from books.toscrape.com, cleans the data,
stores it in SQLite, and performs SQL and Pandas analysis.

--- Cell 2 (code) ---
import requests
import pandas as pd
from bs4 import BeautifulSoup

--- Cell 3 (code) ---
url = "https://books.toscrape.com/"
response = requests.get(url)
print(response.status_code)

--- Cell 4 (code) ---
print(response.text[:500])

--- Cell 5 (code) ---
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)

--- Cell 6 (code) ---
books = soup.find_all("article", class_="product_pod")

print(len(books))

--- Cell 7 (code) ---
first_book = books[0]

print(first_book)

--- Cell 8 (code) ---
title = first_book.h3.a["title"]

print(title)

--- Cell 9 (code) ---
price = first_book.find("p", class_="price_color").text

print(price)

--- Cell 10 (code) ---
rating = first_book.find("p")["class"][1]

print(rating)

--- Cell 11 (code) ---
book_link = first_book.h3.a["href"]

pr

# Data Pipeline – Book Scraping and Analysis

## 1. Data Collection

Collect book details from Books to Scrape using Python, Requests and BeautifulSoup.

In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

all_books = []

for page in range(1, 6):

    if page == 1:
        page_url = "https://books.toscrape.com/"
    else:
        page_url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(page_url, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating = book.find("p")["class"][1]

        detail_url = urljoin(page_url, book.h3.a["href"])

        detail_response = requests.get(detail_url, timeout=15)
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        availability = detail_soup.find(
            "p", class_="instock availability"
        ).text.strip()

        breadcrumb = detail_soup.find("ul", class_="breadcrumb")
        category = breadcrumb.find_all("li")[2].text.strip()

        all_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "category": category
        })

        time.sleep(0.1)

df = pd.DataFrame(all_books)

print("Total books collected:", len(df))
print("Dataset shape:", df.shape)

df.head()

Total books collected: 100
Dataset shape: (100, 5)


,title,price,rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock (22 available),Poetry
1,Tipping the Velvet,Â£53.74,One,In stock (20 available),Historical Fiction
2,Soumission,Â£50.10,One,In stock (20 available),Fiction
3,Sharp Objects,Â£47.82,Four,In stock (20 available),Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock (20 available),History


## 2. Data Exploration

In [6]:
# Check dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   rating        100 non-null    object
 3   availability  100 non-null    object
 4   category      100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


In [7]:
df.isnull().sum()

,0
title,0
price,0
rating,0
availability,0
category,0


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["price"].head(10)

,price
0,Â£51.77
1,Â£53.74
2,Â£50.10
3,Â£47.82
4,Â£54.23
5,Â£22.65
6,Â£33.34
7,Â£17.93
8,Â£22.60
9,Â£52.15


In [10]:
df["price_gbp"] = (
    df["price"]
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
)

df[["price", "price_gbp"]].head()

,price,price_gbp
0,Â£51.77,51.77
1,Â£53.74,53.74
2,Â£50.10,50.10
3,Â£47.82,47.82
4,Â£54.23,54.23


In [11]:
df["price_gbp"] = pd.to_numeric(df["price_gbp"], errors="coerce")

print(df["price_gbp"].dtype)
df[["price_gbp"]].head()

float64


,price_gbp
0,51.77
1,53.74
2,50.10
3,47.82
4,54.23


In [12]:
df["price_gbp"].isnull().sum()

np.int64(0)

In [13]:
df["rating"].value_counts()

,count
rating,
Three,22
One,22
Five,19
Two,19
Four,18


In [14]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

df[["rating"]].head()

,rating
0,3
1,1
2,1
3,4
4,5


In [15]:
print(df["rating"].dtype)

int64


In [16]:
df["in_stock"] = df["availability"].str.contains("In stock")

df[["availability", "in_stock"]].head()

,availability,in_stock
0,In stock (22 available),True
1,In stock (20 available),True
2,In stock (20 available),True
3,In stock (20 available),True
4,In stock (20 available),True


In [17]:
df["in_stock"].value_counts()

,count
in_stock,
True,100


In [18]:
CONVERSION_RATE = 105.50

df["price_inr"] = df["price_gbp"] * CONVERSION_RATE

df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.735
1,53.74,5669.570
2,50.10,5285.550
3,47.82,5045.010
4,54.23,5721.265


In [19]:
df["price_inr"] = df["price_inr"].round(2)

df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.74
1,53.74,5669.57
2,50.10,5285.55
3,47.82,5045.01
4,54.23,5721.26


In [20]:
print("Missing GBP prices:", df["price_gbp"].isnull().sum())
print("Missing INR prices:", df["price_inr"].isnull().sum())
print("Missing ratings:", df["rating"].isnull().sum())
print("Missing stock values:", df["in_stock"].isnull().sum())

Missing GBP prices: 0
Missing INR prices: 0
Missing ratings: 0
Missing stock values: 0


In [21]:
df.head()

,title,price,rating,availability,category,price_gbp,in_stock,price_inr
0,A Light in the Attic,Â£51.77,3,In stock (22 available),Poetry,51.77,True,5461.74
1,Tipping the Velvet,Â£53.74,1,In stock (20 available),Historical Fiction,53.74,True,5669.57
2,Soumission,Â£50.10,1,In stock (20 available),Fiction,50.10,True,5285.55
3,Sharp Objects,Â£47.82,4,In stock (20 available),Mystery,47.82,True,5045.01
4,Sapiens: A Brief History of Humankind,Â£54.23,5,In stock (20 available),History,54.23,True,5721.26


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price         100 non-null    object 
 2   rating        100 non-null    int64  
 3   availability  100 non-null    object 
 4   category      100 non-null    object 
 5   price_gbp     100 non-null    float64
 6   in_stock      100 non-null    bool   
 7   price_inr     100 non-null    float64
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 5.7+ KB


In [50]:
import sqlite3

In [24]:
conn = sqlite3.connect("books.db")

print("Database created successfully!")

Database created successfully!


In [51]:
cursor = conn.cursor()

In [53]:
conn.execute("PRAGMA foreign_keys = ON")

conn.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock BOOLEAN,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")
# Clear previous records before inserting fresh data
conn.execute("DELETE FROM books")
conn.execute("DELETE FROM categories")
conn.commit()

print("Database is ready for fresh data.")

Database is ready for fresh data.


In [54]:
categories = df["category"].unique()

for category in categories:
    conn.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

print("Total unique categories:", len(categories))

Total unique categories: 29


In [55]:
for _, row in df.iterrows():

    category_id = conn.execute(
        "SELECT category_id FROM categories WHERE category_name = ?",
        (row["category"],)
    ).fetchone()[0]

    conn.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        row["in_stock"],
        category_id
    ))

conn.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [28]:
book_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

category_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_categories FROM categories",
    conn
)

print(book_count)
print(category_count)

   total_books
0          100
   total_categories
0                29


In [29]:
query = """
SELECT title, price_gbp, rating
FROM books
WHERE price_gbp > 40
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,title,price_gbp,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5
5,The Black Maria,52.15,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Our Band Could Be Your Life: Scenes from the A...,57.25,3
8,Libertarianism for Beginners,51.33,2
9,It's Only the Himalayas,45.17,2


In [30]:
query = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,title,price_gbp
0,The Death of Humanity: and the Case for Life,58.11
1,Slow States of Collapse: Poems,57.31
2,Our Band Could Be Your Life: Scenes from the A...,57.25
3,The Past Never Ends,56.50
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41
5,Masks and Shadows,56.40
6,The Secret of Dreadwillow Carse,56.13
7,The Electric Pencil: Drawings from Inside Stat...,56.06
8,Birdsong: A Story in Pictures,54.64
9,Sapiens: A Brief History of Humankind,54.23


In [31]:
query = """
SELECT title, rating, price_gbp
FROM books
WHERE rating = 5
LIMIT 5
"""

result = pd.read_sql_query(query, conn)

result

,title,rating,price_gbp
0,Sapiens: A Brief History of Humankind,5,54.23
1,Set Me Free,5,17.46
2,Scott Pilgrim's Precious Little Life (Scott Pi...,5,52.29
3,Rip it Up and Start Again,5,35.02
4,Chase Me (Paris Nights #2),5,25.27


In [32]:
query = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""

result = pd.read_sql_query(query, conn)

result

,rating
0,1
1,2
2,3
3,4
4,5


In [33]:
query = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,title,price_gbp
0,The Inefficiency Assassin: Time Management Tac...,20.59
1,Shakespeare's Sonnets,20.66
2,In the Country We Love: My Family Divided,22.00
3,America's Cradle of Quarterbacks: Western Penn...,22.50
4,The Boys in the Boat: Nine Americans and Their...,22.60
5,The Requiem Red,22.65
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11
7,The Elephant Tree,23.82
8,Olio,23.88
9,The Mindfulness and Acceptance Workbook for An...,23.89


In [34]:
query = """
SELECT
    books.title,
    books.price_gbp,
    categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,title,price_gbp,category_name
0,A Light in the Attic,51.77,Poetry
1,Tipping the Velvet,53.74,Historical Fiction
2,Soumission,50.10,Fiction
3,Sharp Objects,47.82,Mystery
4,Sapiens: A Brief History of Humankind,54.23,History
5,The Requiem Red,22.65,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,Default
9,The Black Maria,52.15,Poetry


In [35]:
books_table = pd.read_sql_query("SELECT * FROM books", conn)

categories_table = pd.read_sql_query(
    "SELECT * FROM categories", conn
)

pandas_result = books_table.merge(
    categories_table,
    on="category_id",
    how="inner"
)

pandas_result[["title", "price_gbp", "category_name"]].head(10)

,title,price_gbp,category_name
0,A Light in the Attic,51.77,Poetry
1,Tipping the Velvet,53.74,Historical Fiction
2,Soumission,50.10,Fiction
3,Sharp Objects,47.82,Mystery
4,Sapiens: A Brief History of Humankind,54.23,History
5,The Requiem Red,22.65,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,Default
9,The Black Maria,52.15,Poetry


In [36]:
sql_result = pd.read_sql_query("""
SELECT books.title, books.price_gbp, categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
""", conn)

pandas_result = pandas_result[
    ["title", "price_gbp", "category_name"]
]

comparison = sql_result.equals(pandas_result)

print("Do SQL JOIN and pandas merge match?", comparison)


Do SQL JOIN and pandas merge match? True


In [37]:
query = """
SELECT
    categories.category_name,
    COUNT(books.book_id) AS total_books
FROM books
JOIN categories
ON books.category_id = categories.category_id
GROUP BY categories.category_name
ORDER BY total_books DESC
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,category_name,total_books
0,Sequential Art,14
1,Nonfiction,12
2,Default,9
3,Poetry,7
4,Food and Drink,5
5,Fiction,5
6,Add a comment,5
7,Young Adult,4
8,History,4
9,Fantasy,4


In [38]:
query = """
SELECT
    COUNT(*) AS total_books,
    ROUND(AVG(price_gbp), 2) AS average_price,
    MIN(price_gbp) AS minimum_price,
    MAX(price_gbp) AS maximum_price
FROM books
"""

result = pd.read_sql_query(query, conn)

result

,total_books,average_price,minimum_price,maximum_price
0,100,34.56,10.16,58.11


In [39]:
query = """
SELECT COUNT(*) AS books_without_category
FROM books
WHERE category_id IS NULL
"""

result = pd.read_sql_query(query, conn)

result

,books_without_category
0,0


In [40]:
query = """
SELECT category_name
FROM categories
ORDER BY category_name
"""

result = pd.read_sql_query(query, conn)

result

,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health


In [41]:
df[df["category"] == "Add a comment"][
    ["title", "category"]
]

,title,category
55,The Torch Is Passed: A Harding Family Story,Add a comment
63,The Mindfulness and Acceptance Workbook for An...,Add a comment
70,The Art Forger,Add a comment
88,On a Midnight Clear,Add a comment
97,Judo: Seven Steps to Black Belt (an Introducto...,Add a comment


In [42]:
target_titles = df.loc[
    df["category"] == "Add a comment", "title"
].tolist()

for page in range(1, 6):

    page_url = (
        "https://books.toscrape.com/"
        if page == 1
        else f"https://books.toscrape.com/catalogue/page-{page}.html"
    )

    soup = BeautifulSoup(
        requests.get(page_url, timeout=15).text,
        "html.parser"
    )

    for book in soup.find_all("article", class_="product_pod"):

        title = book.h3.a["title"]

        if title in target_titles:
            detail_url = urljoin(page_url, book.h3.a["href"])

            detail_soup = BeautifulSoup(
                requests.get(detail_url, timeout=15).text,
                "html.parser"
            )

            breadcrumb = detail_soup.find(
                "ul", class_="breadcrumb"
            )

            print("\nBook:", title)
            print("Breadcrumb:", [
                item.get_text(strip=True)
                for item in breadcrumb.find_all("li")
            ])


Book: The Torch Is Passed: A Harding Family Story
Breadcrumb: ['Home', 'Books', 'Add a comment', 'The Torch Is Passed: A Harding Family Story']

Book: The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy
Breadcrumb: ['Home', 'Books', 'Add a comment', 'The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy']

Book: The Art Forger
Breadcrumb: ['Home', 'Books', 'Add a comment', 'The Art Forger']

Book: On a Midnight Clear
Breadcrumb: ['Home', 'Books', 'Add a comment', 'On a Midnight Clear']

Book: Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)
Breadcrumb: ['Home', 'Books', 'Add a comment', 'Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)']


In [43]:
df["category"] = df["category"].replace(
    "Add a comment", "Unknown"
)

print("Updated category count:")
print(df["category"].value_counts().head(10))

print(
    "\nBooks with Unknown category:",
    (df["category"] == "Unknown").sum()
)

Updated category count:
category
Sequential Art    14
Nonfiction        12
Default            9
Poetry             7
Fiction            5
Food and Drink     5
Unknown            5
Young Adult        4
History            4
Fantasy            4
Name: count, dtype: int64

Books with Unknown category: 5


In [44]:
conn.execute("""
UPDATE categories
SET category_name = 'Unknown'
WHERE category_name = 'Add a comment'
""")

conn.commit()

print("SQLite category updated successfully!")

SQLite category updated successfully!


In [45]:
query = """
SELECT
    categories.category_name,
    COUNT(books.book_id) AS total_books
FROM books
JOIN categories
ON books.category_id = categories.category_id
WHERE categories.category_name = 'Unknown'
GROUP BY categories.category_name
"""

result = pd.read_sql_query(query, conn)

result

,category_name,total_books
0,Unknown,5


In [46]:
query = """
SELECT
    COUNT(*) AS total_books,
    COUNT(DISTINCT category_id) AS total_categories
FROM books
"""

pd.read_sql_query(query, conn)

,total_books,total_categories
0,100,29


In [47]:
conn.execute("PRAGMA foreign_key_check").fetchall()

[]

In [58]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS total_books,
    COUNT(DISTINCT category_id) AS total_categories
FROM books
""", conn)

,total_books,total_categories
0,100,29


In [59]:
conn.execute("PRAGMA foreign_key_check").fetchall()


[]

In [60]:
!ls ai-ml-final-project/data_pipeline

01_data_pipeline.ipynb	README.md


In [62]:
import os

print("Current folder:", os.getcwd())
print("\nAvailable files:")
print(os.listdir("/content"))

Current folder: /content

Available files:
['.config', 'ai-ml-final-project', 'books.db', 'sample_data']
